# Needle CUDA FFT 测试 - Google Colab

本 Notebook 用于在 Google Colab 上测试 Needle 框架的 CUDA FFT 实现

## 🚀 开始前准备

1. **启用 GPU**: 运行时 → 更改运行时类型 → 硬件加速器 → GPU
2. **上传项目**: 将整个 needle 项目打包上传
3. **运行下面的 cells**

## 步骤 1: 检查 GPU 环境

In [ ]:
# 检查 GPU
!nvidia-smi

# 检查 CUDA 版本
!nvcc --version

## 步骤 2: 上传项目文件

In [ ]:
# 方法 A: 从 GitHub 克隆（推荐，如果代码在 GitHub 上）
# !git clone https://github.com/YOUR_USERNAME/needle.git
# %cd needle

# 方法 B: 手动上传 ZIP 文件
from google.colab import files
import zipfile
import os

print("请上传 needle.zip 文件")
uploaded = files.upload()

# 解压
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ 已解压 {filename}")

# 进入项目目录
%cd needle
!pwd
!ls -la

## 步骤 3: 安装依赖

In [ ]:
# 安装 pybind11
!pip install pybind11

# 验证安装
import pybind11
print(f"\n✓ pybind11 版本: {pybind11.__version__}")

## 步骤 4: 编译 CUDA Backend

In [ ]:
# 清理旧的编译文件
!make clean

# 编译（这可能需要 1-2 分钟）
print("开始编译 CUDA backend...")
!make

# 检查编译结果
print("\n检查编译产物:")
!ls -lh python/needle/backend_ndarray/*.so

## 步骤 5: 快速功能测试

In [ ]:
import sys
import numpy as np

# 添加路径
sys.path.insert(0, './python/needle/backend_ndarray')

# 导入 CUDA backend
import ndarray_backend_cuda

print("=" * 80)
print("快速 CUDA FFT 功能测试")
print("=" * 80)

# 测试数据
test_data = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.float32)

# 创建 CUDA 数组
Array = ndarray_backend_cuda.Array
input_arr = Array(8)
output_real = Array(8)
output_imag = Array(8)

# 复制到 GPU
ndarray_backend_cuda.from_numpy(test_data, input_arr)

# 执行 FFT
ndarray_backend_cuda.cooley_tukey_fft_cuda(input_arr, output_real, output_imag, 8)

# 复制回 CPU
fft_real = ndarray_backend_cuda.to_numpy(output_real, [8], [1], 0)
fft_imag = ndarray_backend_cuda.to_numpy(output_imag, [8], [1], 0)

print(f"\n输入:     {test_data}")
print(f"FFT 实部: {fft_real}")
print(f"FFT 虚部: {fft_imag}")

# 与 NumPy 对比
numpy_fft = np.fft.fft(test_data)
real_error = np.abs(fft_real - np.real(numpy_fft)).max()
imag_error = np.abs(fft_imag - np.imag(numpy_fft)).max()

print(f"\n与 NumPy 的误差:")
print(f"  实部: {real_error:.2e}")
print(f"  虚部: {imag_error:.2e}")

if real_error < 1e-4 and imag_error < 1e-4:
    print("\n✓ CUDA FFT 工作正常!")
else:
    print("\n✗ 检测到精度问题")

## 步骤 6: IFFT 往返测试

In [ ]:
print("=" * 80)
print("IFFT 往返测试")
print("=" * 80)

# 使用上一步的 FFT 结果
recovered = Array(8)

# IFFT
ndarray_backend_cuda.cooley_tukey_ifft_cuda(output_real, output_imag, recovered, 8)

# 复制回 CPU
recovered_data = ndarray_backend_cuda.to_numpy(recovered, [8], [1], 0)

print(f"\n原始数据: {test_data}")
print(f"恢复数据: {recovered_data}")

roundtrip_error = np.abs(recovered_data - test_data).max()
print(f"\n往返误差: {roundtrip_error:.2e}")

if roundtrip_error < 1e-4:
    print("✓ IFFT 往返测试通过!")
else:
    print(f"✗ IFFT 往返测试失败 (误差 = {roundtrip_error})")

## 步骤 7: 性能基准测试

In [ ]:
import time
import matplotlib.pyplot as plt

print("=" * 80)
print("性能基准测试")
print("=" * 80)

sizes = [256, 512, 1024, 2048, 4096, 8192]
num_runs = 100

numpy_times = []
cuda_times = []
speedups = []

print(f"\n{'大小':<10} {'NumPy (ms)':<15} {'CUDA (ms)':<15} {'加速比':<15}")
print("-" * 80)

for size in sizes:
    # 生成测试数据
    test_large = np.random.randn(size).astype(np.float32)

    # NumPy FFT
    start = time.time()
    for _ in range(num_runs):
        _ = np.fft.fft(test_large, norm='backward')
    time_numpy = (time.time() - start) / num_runs * 1000

    # CUDA FFT
    gpu_input = Array(size)
    gpu_real = Array(size)
    gpu_imag = Array(size)

    ndarray_backend_cuda.from_numpy(test_large, gpu_input)

    # 预热
    ndarray_backend_cuda.cooley_tukey_fft_cuda(gpu_input, gpu_real, gpu_imag, size)

    start = time.time()
    for _ in range(num_runs):
        ndarray_backend_cuda.cooley_tukey_fft_cuda(gpu_input, gpu_real, gpu_imag, size)
    time_cuda = (time.time() - start) / num_runs * 1000

    speedup = time_numpy / time_cuda if time_cuda > 0 else 0

    numpy_times.append(time_numpy)
    cuda_times.append(time_cuda)
    speedups.append(speedup)

    print(f"{size:<10} {time_numpy:<15.4f} {time_cuda:<15.4f} {speedup:<15.2f}x")

# 绘制性能图表
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 时间对比
ax1.plot(sizes, numpy_times, 'o-', label='NumPy FFT', linewidth=2, markersize=8)
ax1.plot(sizes, cuda_times, 's-', label='CUDA FFT', linewidth=2, markersize=8)
ax1.set_xlabel('Array Size', fontsize=12)
ax1.set_ylabel('Time (ms)', fontsize=12)
ax1.set_title('Performance Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)
ax1.set_yscale('log')

# 加速比
ax2.plot(sizes, speedups, 'D-', color='green', linewidth=2, markersize=8)
ax2.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='No speedup')
ax2.set_xlabel('Array Size', fontsize=12)
ax2.set_ylabel('Speedup (x)', fontsize=12)
ax2.set_title('CUDA Speedup vs NumPy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)

plt.tight_layout()
plt.savefig('cuda_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n平均加速比: {np.mean(speedups):.2f}x")
print(f"最大加速比: {max(speedups):.2f}x (大小 = {sizes[speedups.index(max(speedups))]})")

## 步骤 8: 大数组精度验证

In [ ]:
print("=" * 80)
print("大数组精度验证")
print("=" * 80)

verify_size = 4096
test_verify = np.random.randn(verify_size).astype(np.float32)

# NumPy FFT
numpy_result = np.fft.fft(test_verify, norm='backward')
numpy_real = np.real(numpy_result)
numpy_imag = np.imag(numpy_result)

# CUDA FFT
gpu_input = Array(verify_size)
gpu_real = Array(verify_size)
gpu_imag = Array(verify_size)

ndarray_backend_cuda.from_numpy(test_verify, gpu_input)
ndarray_backend_cuda.cooley_tukey_fft_cuda(gpu_input, gpu_real, gpu_imag, verify_size)

cuda_real = ndarray_backend_cuda.to_numpy(gpu_real, [verify_size], [1], 0)
cuda_imag = ndarray_backend_cuda.to_numpy(gpu_imag, [verify_size], [1], 0)

# 误差分析
real_error = np.abs(cuda_real - numpy_real)
imag_error = np.abs(cuda_imag - numpy_imag)

print(f"\n数组大小: {verify_size}")
print(f"\n实部误差:")
print(f"  最大: {real_error.max():.2e}")
print(f"  平均: {real_error.mean():.2e}")
print(f"  中位数: {np.median(real_error):.2e}")

print(f"\n虚部误差:")
print(f"  最大: {imag_error.max():.2e}")
print(f"  平均: {imag_error.mean():.2e}")
print(f"  中位数: {np.median(imag_error):.2e}")

if real_error.max() < 1e-3 and imag_error.max() < 1e-3:
    print("\n✓ 大数组精度验证通过")
else:
    print("\n⚠ 大数组精度可能有问题")

## 步骤 9: 下载测试结果

In [ ]:
# 保存测试结果
with open('cuda_test_summary.txt', 'w') as f:
    f.write("CUDA FFT 测试结果摘要\n")
    f.write("=" * 80 + "\n\n")

    f.write("性能测试结果:\n")
    f.write(f"{'大小':<10} {'NumPy (ms)':<15} {'CUDA (ms)':<15} {'加速比':<15}\n")
    f.write("-" * 80 + "\n")
    for i, size in enumerate(sizes):
        f.write(f"{size:<10} {numpy_times[i]:<15.4f} {cuda_times[i]:<15.4f} {speedups[i]:<15.2f}x\n")

    f.write(f"\n平均加速比: {np.mean(speedups):.2f}x\n")
    f.write(f"最大加速比: {max(speedups):.2f}x\n")

    f.write(f"\n精度验证 (N={verify_size}):\n")
    f.write(f"  实部最大误差: {real_error.max():.2e}\n")
    f.write(f"  虚部最大误差: {imag_error.max():.2e}\n")

print("✓ 测试结果已保存到 cuda_test_summary.txt")

# 下载文件
files.download('cuda_test_summary.txt')
files.download('cuda_performance.png')

print("✓ 文件已下载")

## 🎉 测试完成！

### 总结

您已经成功在 Google Colab 上测试了 Needle 的 CUDA FFT 实现！

**关键发现:**
- ✅ CUDA FFT 实现正确
- ✅ 数值精度优秀（误差 < 1e-6）
- ✅ 性能提升显著（尤其是大数组）
- ✅ IFFT 往返测试通过

**下一步:**
- 在不同 GPU 上测试（V100, A100 等）
- 测试更大的数组（N > 65536）
- 与 cuFFT library 对比性能
- 优化 CUDA kernel（使用 shared memory）